# TV1 - Tuần 1: Tải MAMA-MIA vào Drive + Step 1-2-3
Chạy trên **CPU runtime**. Notebook gồm: (1) đăng nhập Drive, (2) tải dữ liệu thô từ XNAT (public, không cần tài khoản) vào Drive, (3) Step 1 lọc 1.480 ca + chia Train/Val/Test, (4) Step 2-3 VOI 96³ + 3 kênh -> `.npz`, (5) kiểm tra Data Contract.

**Lưu ý dung lượng:** dữ liệu thô (Pre + Post1 + mask) khoảng **150 GB** trên Drive. Cần Google One đủ dung lượng; có thể xóa `data_real/raw/images` sau khi xử lý xong (`.npz` chỉ khoảng 3,5 GB).
Cell tải có thể chạy lại nhiều lần: file đã tải sẽ được bỏ qua.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q SimpleITK scikit-image scikit-learn pandas openpyxl pyyaml tqdm matplotlib

In [ ]:
import os, sys, re
from pathlib import Path

PROJ = Path('/content/drive/MyDrive/CS406-SCB-TumorSeg')   # thư mục project trên Drive
DATA = 'data_real'                                          # thư mục dữ liệu trong project
WORKERS = os.cpu_count() or 2

os.chdir(PROJ); sys.path.insert(0, str(PROJ))
RAW = PROJ / DATA / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

# config riêng: mọi đường dẫn data/* -> DATA/*
cfg_text = (PROJ / 'configs/default_config.yaml').read_text(encoding='utf-8')
cfg_text = re.sub(r'"data/(raw|processed|splits)', rf'"{DATA}/\1', cfg_text)
(PROJ / 'configs/real_config.yaml').write_text(cfg_text, encoding='utf-8')
print('CPU workers:', WORKERS, '| RAW =', RAW)

## Tải dữ liệu thô từ XNAT vào Drive (chạy lại được)

In [ ]:
import requests, subprocess, shutil, time, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

XNAT = 'https://xnat.health-ri.nl'
SCRATCH = Path('/content/dl_tmp'); SCRATCH.mkdir(exist_ok=True)

def fetch(url, dest):
    """Tải về ổ cục bộ Colab rồi chuyển vào Drive (tránh file dở dang trên Drive)."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return 'skipped'
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = SCRATCH / (dest.parent.name + '__' + dest.name + '.part')
    for attempt in range(6):
        r = subprocess.run(['curl', '-s', '-L', '--fail', '--connect-timeout', '30', '--speed-limit', '20000',
                            '--speed-time', '120', '-o', str(part), url])
        if r.returncode == 0 and part.exists() and part.stat().st_size > 0:
            shutil.move(str(part), str(dest))
            return 'downloaded'
        time.sleep(10 * (attempt + 1))
    raise RuntimeError(f'tải lỗi: {url}')

# bảng lâm sàng
fetch(f'{XNAT}/data/projects/mama_mia/resources/2286069/files/clinical_and_imaging_info.xlsx', RAW / 'clinical_and_imaging_info.xlsx')
df = pd.read_excel(RAW / 'clinical_and_imaging_info.xlsx', sheet_name='dataset_info')
valid = df[df['tumor_subtype'].notna()]['patient_id'].astype(str).tolist()
print('Ca hợp lệ (đã bỏ N/A):', len(valid))

exp = {x['label']: x['ID'] for x in requests.get(f'{XNAT}/data/projects/mama_mia/experiments',
        params={'format': 'json', 'columns': 'ID,label'}, timeout=120).json()['ResultSet']['Result']}

jobs = []
for cid in valid:
    base = f"{XNAT}/data/experiments/{exp[cid]}/scans/1/resources/NIfTI/files"
    jobs += [(f'{base}/{cid}_0000.nii.gz', RAW / 'images' / cid / f'{cid}_0000.nii.gz'),
             (f'{base}/{cid}_0001.nii.gz', RAW / 'images' / cid / f'{cid}_0001.nii.gz'),
             (f'{base}/{cid}_segmentation.nii.gz', RAW / 'segmentations' / 'expert' / f'{cid}.nii.gz')]

DL_WORKERS = 16
failed, counts = [], {'downloaded': 0, 'skipped': 0}
with ThreadPoolExecutor(max_workers=DL_WORKERS) as ex:
    futs = {ex.submit(fetch, u, d): d for u, d in jobs}
    for f in tqdm(as_completed(futs), total=len(futs), desc='Tải'):
        try: counts[f.result()] += 1
        except Exception as e: failed.append(f'{futs[f]}\t{e}')
print(counts, '| lỗi:', len(failed))
if failed: print('\n'.join(failed[:5]), '\n=> chạy lại cell này để thử lại các file lỗi')
# XNAT không có mask automatic và train_test_splits.csv: xử lý ở Step 1 (tự chia Test 20%)

## Step 1 - Lọc ca N/A, chia Stratified Train 85% / Val 15%, giữ nguyên Test

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from src.utils.config import load_config, seed_everything
from src.preprocessing import run_step1_pipeline

cfg = load_config('configs/real_config.yaml'); seed_everything(cfg['project']['seed'])
P = cfg['paths']

splits_csv = Path(P['official_splits_csv'])
if not splits_csv.exists():   # không có split chính thức của bài báo -> tự chia Test 20%
    d = pd.read_excel(P['clinical_excel'], sheet_name='dataset_info')
    d = d[d['tumor_subtype'].notna()]
    key = d['tumor_subtype'].astype(str) + '__' + d['dataset'].astype(str)
    key = key.where(key.map(key.value_counts()) >= 2, d['tumor_subtype'].astype(str))
    tr, te = train_test_split(d['patient_id'], test_size=0.2, random_state=cfg['project']['seed'], stratify=key)
    pd.DataFrame({'train_split': pd.Series(tr.values), 'test_split': pd.Series(te.values)}).to_csv(splits_csv, index=False)
    print('Không thấy train_test_splits.csv -> đã tự chia Test 20% (KHÔNG trùng split của bài báo)')

meta = run_step1_pipeline(P['clinical_excel'], splits_csv, P['splits_metadata_csv'],
                          val_ratio=float(cfg['step1_ingestion']['val_ratio']), seed=int(cfg['project']['seed']))
print('Số ca hợp lệ:', len(meta))
pd.crosstab(meta['tumor_subtype'], meta['split'], margins=True)

## Step 2 + 3 - RAS, 1 mm³, VOI 96³, 3 kênh -> `.npz` (song song, chạy lại được)

In [ ]:
import SimpleITK as sitk, time
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
from src.preprocessing import standardize_and_extract_voi_fast as standardize_and_extract_voi, synthesize_three_channels
sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(1)   # mỗi tiến trình 1 luồng ITK, tránh tranh CPU

NPZ = Path(P['processed_npz_dir']); NPZ.mkdir(parents=True, exist_ok=True)
LOGS = Path(P['logs_dir']); LOGS.mkdir(parents=True, exist_ok=True)

def process_case(case_id, subtype_id):
    out = NPZ / f'{case_id}.npz'
    if out.exists():
        return case_id, 'skipped', None, False
    try:
        s2, s3 = cfg['step2_spatial_voi'], cfg['step3_image_processing']
        img = Path(P['raw_images_dir']) / case_id
        auto = Path(P['raw_auto_seg_dir']) / f'{case_id}.nii.gz'
        pre_voi, post1_voi, exp_voi, auto_voi, _ = standardize_and_extract_voi(
            pre_sitk=sitk.ReadImage(str(img / f'{case_id}_0000.nii.gz'), sitk.sitkFloat32),
            post1_sitk=sitk.ReadImage(str(img / f'{case_id}_0001.nii.gz'), sitk.sitkFloat32),
            expert_mask_sitk=sitk.ReadImage(str(Path(P['raw_expert_seg_dir']) / f'{case_id}.nii.gz'), sitk.sitkUInt8),
            auto_mask_sitk=sitk.ReadImage(str(auto), sitk.sitkUInt8) if auto.exists() else None,
            target_orientation=str(s2['target_orientation']), target_spacing=tuple(float(x) for x in s2['target_spacing']),
            voi_size=tuple(int(x) for x in s2['voi_size']), margin_ratio=float(s2['voi_margin_ratio']), pad_mode=str(s2['pad_mode']))
        image = synthesize_three_channels(
            pre_voi=pre_voi, post1_voi=post1_voi,
            apply_n4=bool(s3['apply_n4_bias_correction']), n4_shrink_factor=int(s3['n4_shrink_factor']),
            apply_denoise=bool(s3['apply_edge_preserving_denoise']), denoise_method=str(s3['denoise_method']),
            nlmeans_patch_size=int(s3['nlmeans_patch_size']), nlmeans_patch_distance=int(s3['nlmeans_patch_distance']),
            nlmeans_h_factor=float(s3['nlmeans_h_factor']), clip_percentiles=tuple(float(x) for x in s3['clip_percentiles']),
            pe_tau_percentile=float(s3['pe_tau_percentile']), clahe_clip_limit=float(s3['clahe_clip_limit']),
            clahe_kernel_size=int(s3['clahe_kernel_size']), zscore_eps=float(s3['zscore_eps']), output_dtype=str(s3['output_dtype']))
        mask = exp_voi[np.newaxis].astype(np.uint8)
        amask = np.zeros_like(mask) if auto_voi is None else auto_voi[np.newaxis].astype(np.uint8)
        tmp = out.with_name(out.stem + '.tmp.npz')
        np.savez_compressed(tmp, image=image, mask=mask, auto_mask=amask, subtype_id=np.int64(subtype_id))
        tmp.replace(out)
        return case_id, 'done', None, auto_voi is None
    except Exception as e:
        return case_id, 'failed', f'{type(e).__name__}: {e}', False

jobs = [(str(r.patient_id), int(r.subtype_id)) for r in meta.itertuples()]
failed, no_auto, counts = [], [], {'done': 0, 'skipped': 0, 'failed': 0}
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(process_case, *j) for j in jobs]
    for f in tqdm(as_completed(futs), total=len(futs)):
        cid, st, err, na = f.result(); counts[st] += 1
        if st == 'failed': failed.append(f'{cid}\t{err}')
        if na: no_auto.append(cid)
print(counts)
if failed: (LOGS / 'preprocess_failed_cases.tsv').write_text('\n'.join(failed), encoding='utf-8'); print('Ca lỗi (chạy lại cell để thử lại):', failed[:5])
if no_auto: (LOGS / 'missing_auto_mask_cases.txt').write_text('\n'.join(no_auto), encoding='utf-8'); print(len(no_auto), 'ca thiếu mask automatic (lưu bằng 0)')

## Kiểm tra Data Contract & QC

In [ ]:
rows, bad = [], []
for pid, sid in zip(meta['patient_id'], meta['subtype_id']):
    f = NPZ / f'{pid}.npz'
    if not f.exists(): bad.append((pid, 'missing')); continue
    z = np.load(f)
    ok = (z['image'].ndim == 4 and z['image'].shape[0] == 3 and z['image'].dtype == np.float16
          and z['mask'].shape == (1,) + z['image'].shape[1:] and int(z['subtype_id']) == int(sid)
          and min(z['image'].shape[1:]) >= 96 and np.isfinite(z['image'].astype(np.float32)).all())
    if not ok: bad.append((pid, 'contract'))
    rows.append((pid, int(z['mask'].sum())))
print(f'Hợp lệ: {len(rows) - len([b for b in bad if b[1] == "contract"])}/{len(meta)} | lỗi: {len(bad)}', bad[:5])
empty = [p for p, v in rows if v == 0]
print('Ca mask rỗng:', len(empty), empty[:5])
print('Dung lượng npz: %.2f GB' % (sum(f.stat().st_size for f in NPZ.glob('*.npz')) / 1e9))

In [ ]:
import matplotlib.pyplot as plt
z = np.load(NPZ / f"{meta['patient_id'].iloc[0]}.npz")
s = int(np.argmax(z['mask'][0].sum(axis=(1, 2))))
fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
for i, t in enumerate(['Post1 (Z)', 'Subtraction (Z)', 'PE+CLAHE (Z)']):
    ax[i].imshow(z['image'][i, s].astype(np.float32), cmap='gray'); ax[i].contour(z['mask'][0, s], colors='r', linewidths=.8); ax[i].set_title(t); ax[i].axis('off')
ax[3].imshow(z['mask'][0, s], cmap='gray'); ax[3].set_title('Expert mask'); ax[3].axis('off')
plt.tight_layout(); plt.show()